# Grader Agent Evaluation

First install the necessary libaries and set OpenAI key.

In [1]:
!pip install openai

In [2]:
from openai import OpenAI
import ast
import csv
import pandas as pd
import time

In [ ]:
client = OpenAI(api_key="sk-proj-")

Next create the grader prompt.

In [4]:
grading_prompt = """
You are a strict grading assistant. Grade the student's answer as either "correct" or "incorrect."

Rules:
- "correct" = The answer includes ALL essential concepts required for a complete and correct response. No key idea is missing.
- "incorrect" = The answer is missing ANY required element, is incomplete, oversimplified, or contains factual or reasoning errors.

Critical rule:
If ANY required element of the correct reasoning is missing, the grade MUST be "incorrect."

Return ONLY: correct or incorrect.
"""

Create a function to grade question and answer pair.

In [10]:
def grade_with_gpt(question, answer):
    payload = f"Question: {question}\nAnswer: {answer}"
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": grading_prompt},
            {"role": "user", "content": payload}
        ]
    )
    raw = response.choices[0].message.content.strip()

    try:
        return raw
    except:
        print(f"Parse error: '{raw}'")
        return None

Get all the data from the dataset.

In [11]:
questions = []
answers = []
actual_grades = []

with open("questions_answers_graded.csv", "r", encoding="utf-8", errors="replace") as f:
    reader = csv.DictReader(f)
    for row in reader:
        questions.append(row["question"])
        answers.append(row["answer"])
        actual_grades.append((row["grade"]))

Run grader agent over dataset.

In [13]:
predicted_grades = []

for q, a in zip(questions, answers):
    g = grade_with_gpt(q, a)
    predicted_grades.append(g)
    time.sleep(1.1)

Print accuracy and any misclassifications.

In [14]:
correct = 0
mismatches = []

for i, (y_true, y_pred) in enumerate(zip(actual_grades, predicted_grades)):
    if y_true == y_pred:
        correct += 1
    else:
        mismatches.append({
            "index": i,
            "question": questions[i],
            "answer": answers[i],
            "actual": y_true,
            "predicted": y_pred
        })

accuracy = correct / len(actual_grades)

print(f"Accuracy: {accuracy:.4f}")
print(f"Total mismatches: {len(mismatches)}\n")

print("--- Examples of mismatches ---")
for m in mismatches[:10]:
    print(f"\nRow {m['index']}:")
    print(f"  Q: {m['question']}")
    print(f"  A: {m['answer']}")
    print(f"  Actual:    {m['actual']}")
    print(f"  Predicted: {m['predicted']}")

Accuracy: 0.9474
Total mismatches: 2

--- Examples of mismatches ---

Row 34:
  Q: Describe why certain regions of nervous tissue show stronger and more uniform toluidine blue staining for myelin?
  A: Because white matter contains dense, evenly packed myelinated axons, it binds more dye, producing stronger and more uniform staining.
  Actual:    correct
  Predicted: incorrect

Row 36:
  Q: How does the spine adjust when carrying a heavy load in a backpack?
  A: The spine extends (leans backward) to counterbalance the weight pulling the body forward, helping keep the center of gravity over the feet.
  Actual:    correct
  Predicted: incorrect
